# Exercise A: NAIRR Models and HuggingFace

**Estimated time:** ~30 minutes

NAIRR provides free pre-trained models through its [Pilot Resources page](https://nairrpilot.org/pilotresources). Some models link directly to HuggingFace, while others require separate registration with the resource provider.

In this exercise, we will use a **NASA-developed sentence transformer** to perform semantic similarity search on scientific text. This demonstrates:

- How to access NAIRR-affiliated models from HuggingFace
- How sentence embeddings enable semantic search across research domains
- How encoding speed scales with corpus size
- **Timing differences between your local machine and Jetstream2**

Browse the available resources: https://nairrpilot.org/pilotresources

In [ ]:
!pip install sentence-transformers torch pandas numpy --quiet

In [ ]:
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util


class timed:
    """Context manager for timing code blocks."""

    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start
        print(f"[{self.label}] {self.elapsed:.2f} seconds")

## About the NASA Indus-Retriever Model

**Model:** [`nasa-impact/nasa-smd-ibm-st-v2`](https://huggingface.co/nasa-impact/nasa-smd-ibm-st-v2)

This is a sentence transformer fine-tuned on NASA Science Mission Directorate (SMD) documents. It is based on IBM's INDUS model and was trained on millions of domain-specific examples to improve scientific information retrieval.

- **Parameters:** ~125M
- **Use case:** Semantic search over scientific text, retrieval-augmented generation
- **Available on HuggingFace:** Free to download and use

This is one of the models accessible through NAIRR's free resources — no allocation required.

In [ ]:
with timed("Model download and load"):
    model = SentenceTransformer("nasa-impact/nasa-smd-ibm-st-v2")

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

## Building a Scientific Corpus

We create a corpus of scientific text passages spanning multiple research domains, then query against them to find semantically related passages. This simulates searching a lab's paper collection or a domain-specific document store.

In [ ]:
corpus = [
    # Climate / Earth Science
    "Sea surface temperature measurements from satellite-based infrared radiometers provide critical data for monitoring ocean heat content. These observations are essential for tracking El Nino events and long-term climate trends.",
    "The Greenland ice sheet has been losing mass at an accelerating rate over the past two decades. Gravity recovery and climate experiment (GRACE) satellite data reveal cumulative losses exceeding 4,000 gigatons since 2002.",
    "Global mean sea level has risen approximately 3.6 millimeters per year since 2006, driven by thermal expansion of ocean water and melting of land-based ice. Coastal communities face increasing flood risk.",
    "Atmospheric CO2 concentrations have surpassed 420 parts per million, the highest level in at least 800,000 years. Carbon cycle models project continued increases under current emission trajectories.",
    # Astrophysics
    "The James Webb Space Telescope has detected atmospheric signatures of carbon dioxide on exoplanet WASP-39b. Transit spectroscopy at infrared wavelengths enables characterization of planetary atmospheres beyond our solar system.",
    "Observations of galaxy rotation curves provide compelling evidence for dark matter. The discrepancy between visible mass and gravitational effects suggests approximately 85 percent of matter in the universe is non-luminous.",
    "Cosmic microwave background anisotropy measurements from the Planck satellite constrain the age of the universe to 13.8 billion years. These observations also confirm the spatially flat geometry predicted by inflationary models.",
    "Fast radio bursts are millisecond-duration radio transients of extragalactic origin. Their dispersion measures provide a novel tool for probing the distribution of baryonic matter in the intergalactic medium.",
    # Materials Science
    "Metal-organic frameworks exhibit exceptionally high surface areas exceeding 7,000 square meters per gram. These porous crystalline materials show promise for carbon capture, gas storage, and catalysis applications.",
    "Perovskite solar cells have achieved power conversion efficiencies above 25 percent in laboratory settings. Stability under operational conditions remains a key challenge for commercial deployment.",
    "High-entropy alloys composed of five or more principal elements in near-equimolar ratios exhibit remarkable mechanical properties including high strength and fracture toughness at cryogenic temperatures.",
    # Biomedicine
    "CRISPR-Cas9 gene editing has enabled precise modification of disease-associated genetic variants in patient-derived cells. Clinical trials for sickle cell disease and beta-thalassemia have shown promising therapeutic outcomes.",
    "AlphaFold2 predicted protein structures with atomic-level accuracy for over 200 million proteins. This breakthrough accelerates drug target identification and rational drug design across therapeutic areas.",
    "Single-cell RNA sequencing reveals previously unknown heterogeneity within tumor microenvironments. This technology enables identification of rare cell populations that drive treatment resistance.",
    # Computer Science / AI
    "Transformer architectures use self-attention mechanisms to capture long-range dependencies in sequential data. Scaling these models to billions of parameters has produced emergent capabilities in language understanding and generation.",
    "Reinforcement learning from human feedback aligns large language models with human preferences. This approach reduces harmful outputs while preserving the model's general capabilities.",
    "Federated learning enables training machine learning models across distributed datasets without centralizing sensitive data. This approach addresses privacy concerns in healthcare and financial applications.",
    # Environmental Science
    "Ocean acidification caused by absorption of anthropogenic CO2 reduces carbonate ion concentrations critical for shell-forming marine organisms. Coral reefs and shellfish populations are particularly vulnerable.",
    "Wildfire modeling using coupled atmosphere-fire simulations improves prediction of fire spread under complex terrain and wind conditions. These models integrate satellite-derived fuel moisture data for operational forecasting.",
    "Microplastic contamination has been detected in deep ocean sediments, Arctic sea ice, and remote mountain soils. Particles smaller than 5 millimeters enter food webs through ingestion by marine organisms.",
]

print(f"Corpus: {len(corpus)} passages")

In [ ]:
with timed("Encode corpus (20 passages)"):
    corpus_embeddings = model.encode(corpus, convert_to_tensor=True, show_progress_bar=True)

print(f"Embedding shape: {corpus_embeddings.shape}")

In [ ]:
queries = [
    "What satellites measure sea surface temperature?",
    "How do transformer architectures handle long sequences?",
    "What are the effects of ocean acidification on marine ecosystems?",
    "How can machine learning accelerate drug discovery?",
    "What evidence supports the existence of dark matter?",
]

with timed("Encode queries"):
    query_embeddings = model.encode(queries, convert_to_tensor=True)

with timed("Compute similarities"):
    cos_scores = util.cos_sim(query_embeddings, corpus_embeddings)

# Display top-3 results for each query
for i, query in enumerate(queries):
    print(f"\n{'=' * 80}")
    print(f"Query: {query}")
    print(f"{'=' * 80}")
    scores = cos_scores[i].cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:3]
    for rank, idx in enumerate(top_indices, 1):
        print(f"  #{rank} (score: {scores[idx]:.4f})")
        print(f"  {corpus[idx][:150]}...")

## Scaling Up

With 20 passages, everything is nearly instant. What happens when the corpus grows to 1,000 or 5,000 passages? This is where compute resources start to matter — and where the difference between your laptop and Jetstream2 becomes visible.

In [ ]:
# Scale to 1,000 passages
large_corpus = corpus * 50
print(f"Scaled corpus: {len(large_corpus)} passages")

with timed("Encode 1,000 passages"):
    large_embeddings = model.encode(
        large_corpus, convert_to_tensor=True, batch_size=64, show_progress_bar=True
    )

with timed("Similarity search (5 queries x 1,000 passages)"):
    large_scores = util.cos_sim(query_embeddings, large_embeddings)

print(f"\nResults matrix shape: {large_scores.shape}")

In [ ]:
# Scale to 5,000 passages
xlarge_corpus = corpus * 250
print(f"Scaled corpus: {len(xlarge_corpus)} passages")

with timed("Encode 5,000 passages"):
    xlarge_embeddings = model.encode(
        xlarge_corpus, convert_to_tensor=True, batch_size=64, show_progress_bar=True
    )

with timed("Similarity search (5 queries x 5,000 passages)"):
    xlarge_scores = util.cos_sim(query_embeddings, xlarge_embeddings)

print(f"\nResults matrix shape: {xlarge_scores.shape}")

## Timing Comparison

| Operation                        | Local (your laptop) | Jetstream2 |
|----------------------------------|--------------------:|-----------:|
| Model load                       |          ___ sec    |   ___ sec  |
| Encode 20 passages               |          ___ sec    |   ___ sec  |
| Encode 1,000 passages            |          ___ sec    |   ___ sec  |
| Encode 5,000 passages            |          ___ sec    |   ___ sec  |
| Similarity (5q x 1,000p)         |          ___ sec    |   ___ sec  |
| Similarity (5q x 5,000p)         |          ___ sec    |   ___ sec  |

**Fill in both columns.** Run this notebook locally first, record the times. Then run the same notebook on Jetstream2 and record those times.

## Why This Matters for Research

- At 5,000 documents the timing difference becomes noticeable
- At 50,000+ documents (a realistic research corpus), the gap becomes a bottleneck
- If you are building a search tool for your lab's paper collection, doing retrieval-augmented generation, or processing a domain-specific corpus, this timing difference translates directly into **research iteration speed**
- With NAIRR compute, you can scale without worrying about your laptop's limitations

## Try It Yourself

- Try encoding a paragraph from your own research abstract
- Try changing the queries to match your domain
- Try a different model from HuggingFace (e.g., `all-MiniLM-L6-v2` for comparison)

In [ ]:
# Try your own research text
my_text = "Replace this with a paragraph from your research."
my_embedding = model.encode([my_text], convert_to_tensor=True)

# Find the most similar passages in our corpus
scores = util.cos_sim(my_embedding, corpus_embeddings)[0].cpu().numpy()
top_idx = np.argsort(scores)[::-1][:3]

print("Most similar passages to your text:")
for rank, idx in enumerate(top_idx, 1):
    print(f"\n  #{rank} (score: {scores[idx]:.4f})")
    print(f"  {corpus[idx][:200]}")